In [1]:
import pandas as pd
import numpy as np
from transformers import pipeline
from shared import load_data, evaluate_model
import ast
from sentence_transformers import SentenceTransformer
from sklearn.preprocessing import normalize

train_df, test_df, restaurants_df = load_data()

C:\Users\nickl\PycharmProjects\PythonProject\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [3]:
from transformers import AutoTokenizer, AutoModelForSequenceClassification
import torch
from tqdm import tqdm

MODEL_NAME = "distilbert-base-uncased-finetuned-sst-2-english"
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
model = AutoModelForSequenceClassification.from_pretrained(MODEL_NAME)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")

model = model.to(device)
model.eval()

BATCH_SIZE = 128


def get_batched_sentiment(df, batch_size=BATCH_SIZE):
    texts = df['text'].fillna("").astype(str).tolist()
    id2label = model.config.id2label
    results = []

    with torch.no_grad():
        for i in tqdm(range(0, len(texts), batch_size), desc="Processing Sentiment"):
            batch_texts = texts[i : i + batch_size]
            inputs = tokenizer(
                batch_texts,
                truncation=True,
                max_length=512,
                padding=True,
                return_tensors="pt"
            )
            inputs = {k: v.to(device) for k, v in inputs.items()}

            outputs = model(**inputs)
            probs = torch.softmax(outputs.logits, dim=-1)
            preds = torch.argmax(probs, dim=-1)
            scores = probs[range(len(preds)), preds].tolist()
            labels = [id2label[p.item()] for p in preds]

            for label, score in zip(labels, scores):
                results.append(score if label == "POSITIVE" else -score)

    return results

train_df['sentiment_score'] = get_batched_sentiment(train_df)

restaurant_sentiment = (
    train_df.groupby('business_id')['sentiment_score']
    .mean()
    .reset_index()
    .rename(columns={'sentiment_score': 'avg_sentiment'})
)

restaurants_df = restaurants_df.merge(restaurant_sentiment, on='business_id', how='left')
restaurants_df['avg_sentiment'] = restaurants_df['avg_sentiment'].fillna(0.0)

Loading weights: 100%|██████████| 104/104 [00:00<00:00, 13488.59it/s]


Using device: cuda


Processing Sentiment: 100%|██████████| 325/325 [03:39<00:00,  1.48it/s]


In [4]:
BOOLEAN_ATTRIBUTES = {
    'RestaurantsTakeOut': 'TakeOut',
    'OutdoorSeating': 'Outdoor',
    'RestaurantsDelivery': 'Deliv',
    'GoodForKids': 'GFK'
}

CATEGORICAL_ATTRIBUTES = {
    'RestaurantsPriceRange2': 'Price',
    'Ambience': 'Amb'
}

ALL_ATTRIBUTES = {**BOOLEAN_ATTRIBUTES, **CATEGORICAL_ATTRIBUTES}

def parse_attributes(attr_str):
    if pd.isna(attr_str): return {}
    return ast.literal_eval(attr_str)

def get_attr(row, attr_name):
    attrs = row.get('parsed_attributes', {})
    if not isinstance(attrs, dict): return 'Unknown'
    return str(attrs.get(attr_name, 'Unknown')).replace("u'", "").replace("'", "")

restaurants_df['parsed_attributes'] = restaurants_df['attributes'].apply(parse_attributes)

for attributes, clean_name in ALL_ATTRIBUTES.items():
    restaurants_df[clean_name] = restaurants_df.apply(lambda row: get_attr(row, attributes), axis=1)

#bin encoding
binary_clean_names = list(BOOLEAN_ATTRIBUTES.values())
binary_feature_cols = []
for col in binary_clean_names:
    bin_name = col + '_Bin'
    restaurants_df[bin_name] = restaurants_df[col].map({'True': 1, 'False': 0, 'Unknown': 0, 'None': 0}).fillna(0).astype(int)
    binary_feature_cols.append(bin_name)

#one hot encoding for attributes
categorical_clean_names = list(CATEGORICAL_ATTRIBUTES.values())

if categorical_clean_names:
    attr_dummies = pd.get_dummies(restaurants_df[categorical_clean_names], dtype=int)
else:
    attr_dummies = pd.DataFrame(index=restaurants_df.index)

#multi hot encoding for categories
restaurants_df['cat_list'] = restaurants_df['categories'].astype(str).apply(lambda x: [c.strip().lower() for c in x.split(',')])
categories_dummies = pd.get_dummies(restaurants_df['cat_list'].explode(), dtype=int).groupby(level=0).sum()

#weights
weight_binary = 0.30
weight_attr = 0.50
weight_cat = 0.20

#concat/normalize
feature_df = pd.concat([
    (restaurants_df[binary_feature_cols] * weight_binary),
    (attr_dummies * weight_attr),
    (categories_dummies * weight_cat)
], axis=1)

feature_matrix_normalized = normalize(feature_df.values, norm='l2', axis=1)
item_indices = pd.Series(restaurants_df.index, index=restaurants_df['business_id']).to_dict()

In [5]:
K_REVIEWS = 20
MIN_WORDS = 8

print(f"Selecting Top {K_REVIEWS} most recent quality reviews per restaurant...")

quality_reviews = train_df.dropna(subset=['text']).copy()
quality_reviews['datetime'] = pd.to_datetime(quality_reviews['datetime'])
quality_reviews['word_count'] = quality_reviews['text'].str.split().str.len()
quality_reviews = quality_reviews[quality_reviews['word_count'] >= MIN_WORDS]

quality_reviews = quality_reviews.sort_values(['business_id', 'datetime'], ascending=[True, False])

top_k_grouped = quality_reviews.groupby('business_id').head(K_REVIEWS)

reviews_final = top_k_grouped.groupby('business_id')['text'].apply(lambda x: ' '.join(x.astype(str))).reset_index()
reviews_final.rename(columns={'text': 'concat_review'}, inplace=True)

restaurants_df = restaurants_df.merge(reviews_final, on='business_id', how='left')
restaurants_df['concat_review'] = restaurants_df['concat_review'].fillna("")

texts_to_encode = restaurants_df['concat_review'].apply(lambda x: ' '.join(x.split()[:380])).tolist()
model = SentenceTransformer('all-mpnet-base-v2')
text_embeddings = model.encode(texts_to_encode, show_progress_bar=True)

text_matrix_normalized = normalize(text_embeddings, norm='l2', axis=1) #L2 NORMALIZING!

print(f"Matrix shape: {text_matrix_normalized.shape}")

Selecting Top 20 most recent quality reviews per restaurant...


C:\Users\nickl\PycharmProjects\PythonProject\.venv\Lib\site-packages\huggingface_hub\file_download.py:138: UserWarning: `huggingface_hub` cache-system uses symlinks by default to efficiently store duplicated files but your machine does not support them in C:\Users\nickl\.cache\huggingface\hub\models--sentence-transformers--all-mpnet-base-v2. Caching files will still work but in a degraded version that might require more space on your disk. This warning can be disabled by setting the `HF_HUB_DISABLE_SYMLINKS_WARNING` environment variable. For more details, see https://huggingface.co/docs/huggingface_hub/how-to-cache#limitations.
To support symlinks on Windows, you either need to activate Developer Mode or to run Python as an administrator. In order to activate developer mode, see this article: https://docs.microsoft.com/en-us/windows/apps/get-started/enable-your-device-for-development
  warnings.warn(message)
Batches: 100%|██████████| 24/24 [00:06<00:00,  3.43it/s]

Matrix shape: (767, 768)


In [6]:
all_train = train_df[train_df['stars'] >= 0.0]

user_meta_profiles = {}
user_text_profiles = {}
user_sentiment_profiles = {}

for user, group in all_train.groupby('user_id'):
    liked_item_ids = group['business_id'].tolist()
    liked_indices = [item_indices[biz] for biz in liked_item_ids if biz in item_indices]

    if liked_indices:
        user_meta_profiles[user] = np.asarray(feature_matrix_normalized[liked_indices].mean(axis=0))
        user_text_profiles[user] = np.asarray(text_matrix_normalized[liked_indices].mean(axis=0))
        user_sentiment_profiles[user] = restaurants_df.iloc[liked_indices]['avg_sentiment'].mean()

In [13]:
from sklearn.metrics.pairwise import cosine_similarity

test_users = test_df['user_id'].unique()
all_businesses = restaurants_df['business_id'].tolist()
item_sentiments = restaurants_df['avg_sentiment'].values
predictions = {}

w_meta = 0.60
w_text = 0.15
w_sentiment = 0.25

for user in test_users:
    if user in user_meta_profiles:
        sim_meta = cosine_similarity(user_meta_profiles[user].reshape(1, -1), feature_matrix_normalized).flatten()

        sim_text = cosine_similarity(user_text_profiles[user].reshape(1, -1), text_matrix_normalized).flatten()

        sim_sentiment = 1 - (np.abs(user_sentiment_profiles[user] - item_sentiments) / 2)

        final_sim_scores = (w_meta * sim_meta) + (w_text * sim_text) + (w_sentiment * sim_sentiment)

        top_indices = final_sim_scores.argsort()[-30:][::-1]
        predictions[user] = [all_businesses[i] for i in top_indices]
    else:
        predictions[user] = []

In [14]:
metrics = evaluate_model(predictions, test_df)
results_df = pd.DataFrame([metrics]).round(4)
results_df.index = [f'Content-Based B (Meta={w_meta}, Text={w_text}, Sentiment={w_sentiment})']
display(results_df)

,Hit@10,Hit@20,Hit@30,NDCG@10,NDCG@20,NDCG@30
"Content-Based B (Meta=0.6, Text=0.15, Sentiment=0.25)",0.0419,0.0792,0.1125,0.0168,0.0261,0.0332
